# `15 — Dynamic RSQ/RMQ: Fenwick tree + Segment tree`

## **Complexity summary**
| Structure | Preproc | Query | Update | Supports |
|---|---:|---:|---:|---|
| **Fenwick tree** | O(N log N) | O(log N) | O(log N) | **RSQ** only |
| **Segment tree** | O(N) | O(log N) | O(log N) | **RSQ and RMQ** |

Key idea:
- Fenwick relies on "inverse" (prefix sums), so it naturally fits **sum**, not **min**.
- Segment tree stores an aggregate (sum or min) per interval node.

In [2]:
# Fenwick tree with binary-step:
from dataclasses import dataclass
from typing import List, Self


def lsb_len(i: int) -> int:
    # Lecture formula: l(i) = ~i & (i+1)
    return (~i) & (i + 1)


@dataclass
class FenwickRSQ:
    n: int
    f: List[int]

    @classmethod
    def build_empty(cls, *, n: int) -> "FenwickRSQ":
        return cls(n=n, f=[0] * n)

    def update_add(self: Self, *, i: int, delta: int, verbose: bool = True) -> None:
        if verbose:
            print("-" * 70)
            print(f"Fenwick update_add(i={i}, delta={delta})")
            print("-" * 70)

        while i < self.n:
            if verbose:
                print(f"  add to f[{i}] (was {self.f[i]}) -> {self.f[i] + delta}")
            self.f[i] += delta
            step = lsb_len(i)
            if verbose:
                print(f"  next i = {i} + lsb_len({i})={step} -> {i + step}")
                print(f"  f now: {self.f}")
            i += step

    def query_prefix(self: Self, *, i: int, verbose: bool = True) -> int:
        # Returns sum a[0..i] inclusive
        res: int = 0
        if verbose:
            print("-" * 70)
            print(f"Fenwick query_prefix(i={i}) = RSQ(0, i+1)")
            print("-" * 70)

        while i >= 0:
            if verbose:
                print(f"  take f[{i}]={self.f[i]} -> res={res + self.f[i]}")
            res += self.f[i]
            step = lsb_len(i)
            if verbose:
                print(f"  next i = {i} - lsb_len({i})={step} -> {i - step}")
            i -= step

        if verbose:
            print(f"Answer prefix sum = {res}")
        return res

    def rsq(self: Self, *, l: int, r: int, verbose: bool = True) -> int:
        # sum over [l, r)
        right: int = self.query_prefix(i=r - 1, verbose=verbose) if r > 0 else 0
        left: int = self.query_prefix(i=l - 1, verbose=verbose) if l > 0 else 0
        res: int = right - left
        if verbose:
            print("-" * 70)
            print(f"RSQ({l}, {r}) = pref({r-1}) - pref({l-1}) = {right} - {left} = {res}")
        return res


# Example: Build Fenwick by updates (as in lecture: build is O(N log N))
a = [3, 5, 10, 1, 6, 8, 9, 8]
fw = FenwickRSQ.build_empty(n=len(a))

print("-" * 70)
print("Building Fenwick via update_add (watch f array change)")
print("-" * 70)
for idx, val in enumerate(a):
    fw.update_add(i=idx, delta=val, verbose=True)

_ = fw.rsq(l=1, r=6, verbose=True)

print("-" * 70)
print("Single update: a[3] += 7 (delta=+7)")
print("-" * 70)
fw.update_add(i=3, delta=7, verbose=True)
_ = fw.rsq(l=1, r=6, verbose=True)

----------------------------------------------------------------------
Building Fenwick via update_add (watch f array change)
----------------------------------------------------------------------
----------------------------------------------------------------------
Fenwick update_add(i=0, delta=3)
----------------------------------------------------------------------
  add to f[0] (was 0) -> 3
  next i = 0 + lsb_len(0)=1 -> 1
  f now: [3, 0, 0, 0, 0, 0, 0, 0]
  add to f[1] (was 0) -> 3
  next i = 1 + lsb_len(1)=2 -> 3
  f now: [3, 3, 0, 0, 0, 0, 0, 0]
  add to f[3] (was 0) -> 3
  next i = 3 + lsb_len(3)=4 -> 7
  f now: [3, 3, 0, 3, 0, 0, 0, 0]
  add to f[7] (was 0) -> 3
  next i = 7 + lsb_len(7)=8 -> 15
  f now: [3, 3, 0, 3, 0, 0, 0, 3]
----------------------------------------------------------------------
Fenwick update_add(i=1, delta=5)
----------------------------------------------------------------------
  add to f[1] (was 3) -> 8
  next i = 1 + lsb_len(1)=2 -> 3
  f now: [3, 8, 

In [4]:
# Segment tree (supports RSQ and RMQ) with tree + query recursion:
from dataclasses import dataclass
from typing import List, Self, Literal


@dataclass
class SegmentTree:
    n: int
    size: int
    tree: List[int]
    mode: Literal["sum", "min"]

    @classmethod
    def build(cls, *, a: List[int], mode: Literal["sum", "min"], verbose: bool = True) -> "SegmentTree":
        n: int = len(a)
        size: int = 1

        while size < n:
            size *= 2

        if mode == "sum":
            neutral: int = 0
            combine = lambda x, y: x + y
        else:
            neutral = 10**18
            combine = lambda x, y: x if x < y else y

        tree: List[int] = [neutral] * (2 * size - 1)

        # Leaves start at index (size - 1)
        for i in range(n):
            tree[size - 1 + i] = a[i]
        for i in range(size - 2, -1, -1):
            tree[i] = combine(tree[2 * i + 1], tree[2 * i + 2])

        st = cls(n=n, size=size, tree=tree, mode=mode)

        if verbose:
            print("-" * 70)
            print(f"Segment tree build (mode={mode})")
            print("-" * 70)
            st.show_levels(label="Tree levels (index:value)")
        return st

    def _neutral(self: Self) -> int:
        return 0 if self.mode == "sum" else 10**18

    def _combine(self: Self, x: int, y: int) -> int:
        return (x + y) if self.mode == "sum" else (x if x < y else y)

    def show_levels(self: Self, *, label: str = "") -> None:
        if label:
            print(label)

        i: int = 0
        level: int = 0
        total: int = len(self.tree)

        while i < total:
            count: int = 2 ** level
            row_idx = list(range(i, min(i + count, total)))
            row = [f"({j}:{self.tree[j]})" for j in row_idx]
            print(f"level {level}: " + "  ".join(row))
            i += count
            level += 1

    def update_set(self: Self, *, idx: int, value: int, verbose: bool = True) -> None:
        if verbose:
            print("-" * 70)
            print(f"update_set(idx={idx}, value={value})")
            print("-" * 70)

        i: int = (self.size - 1) + idx
        self.tree[i] = value

        if verbose:
            print(f"  set leaf tree[{i}] = {value}")

        while i > 0:
            i = (i - 1) // 2
            new_val = self._combine(self.tree[2 * i + 1], self.tree[2 * i + 2])
            if verbose:
                print(f"  refresh node {i}: combine(children) -> {new_val}")
            self.tree[i] = new_val

    def query(self: Self, *, l: int, r: int, verbose: bool = True) -> int:
        if verbose:
            print("-" * 70)
            print(f"query([{l}, {r})) mode={self.mode}")
            print("-" * 70)

        def rec(i: int, li: int, ri: int, depth: int = 0) -> int:
            indent = "  " * depth

            if r <= li or ri <= l:
                if verbose:
                    print(f"{indent}node {i} range=[{li},{ri}) -> disjoint -> return neutral")
                return self._neutral()

            if l <= li and ri <= r:
                if verbose:
                    print(f"{indent}node {i} range=[{li},{ri}) -> fully inside -> return tree[{i}]={self.tree[i]}")
                return self.tree[i]

            mid: int = li + (ri - li) // 2
            if verbose:
                print(f"{indent}node {i} range=[{li},{ri}) -> split at {mid}")

            left_val: int = rec(2 * i + 1, li, mid, depth + 1)
            right_val: int = rec(2 * i + 2, mid, ri, depth + 1)
            res: int = self._combine(left_val, right_val)

            if verbose:
                print(f"{indent}combine -> {res}")
            return res

        return rec(0, 0, self.size)


a = [3, 5, 10, 1, 6, 8, 9, 8]

# RSQ segment tree
st_sum = SegmentTree.build(a=a, mode="sum", verbose=True)
ans_sum = st_sum.query(l=1, r=6, verbose=True)
print("RSQ(1,6) =", ans_sum)

st_sum.update_set(idx=3, value=8, verbose=True)
ans_sum2 = st_sum.query(l=1, r=6, verbose=True)
print("RSQ(1,6) after update =", ans_sum2)

# RMQ segment tree
st_min = SegmentTree.build(a=a, mode="min", verbose=True)
ans_min = st_min.query(l=1, r=6, verbose=True)
print("RMQ(1,6) =", ans_min)

----------------------------------------------------------------------
Segment tree build (mode=sum)
----------------------------------------------------------------------
Tree levels (index:value)
level 0: (0:50)
level 1: (1:19)  (2:31)
level 2: (3:8)  (4:11)  (5:14)  (6:17)
level 3: (7:3)  (8:5)  (9:10)  (10:1)  (11:6)  (12:8)  (13:9)  (14:8)
----------------------------------------------------------------------
query([1, 6)) mode=sum
----------------------------------------------------------------------
node 0 range=[0,8) -> split at 4
  node 1 range=[0,4) -> split at 2
    node 3 range=[0,2) -> split at 1
      node 7 range=[0,1) -> disjoint -> return neutral
      node 8 range=[1,2) -> fully inside -> return tree[8]=5
    combine -> 5
    node 4 range=[2,4) -> fully inside -> return tree[4]=11
  combine -> 16
  node 2 range=[4,8) -> split at 6
    node 5 range=[4,6) -> fully inside -> return tree[5]=14
    node 6 range=[6,8) -> disjoint -> return neutral
  combine -> 14
combine ->

**Real-world mapping**:

- Segment tree is the “Swiss army knife” for range queries with updates:
- min temperature in a time window (RMQ)
- sum of sales in a time window (RSQ)
- both with corrections/updates
